# 03 - Overfitting, Underfitting & Learning Curves

Notebook 01 reported a single test score. A single number cannot tell you *why* a model
is bad - too simple, too complex, or just starved of data. This notebook gives you the
two diagnostic plots that answer that question.

By the end you will be able to:
1. Read the train-vs-test gap as a symptom.
2. Draw a **validation curve** to find the right model complexity.
3. Draw a **learning curve** to decide whether more data would help.
4. Say what bias and variance mean, in terms you can see on a plot.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
np.random.seed(42)

heart = pd.read_csv("../data/heart-disease.csv")
X = heart.drop("target", axis=1)
y = heart["target"]
print(X.shape, y.value_counts().to_dict())
heart.head()

## 1. The gap is the signal

Train a decision tree with no depth limit. It will memorise the training set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

for depth in [1, 3, 5, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    tr, te = tree.score(X_train, y_train), tree.score(X_test, y_test)
    label = "unlimited" if depth is None else str(depth)
    print(f"max_depth={label:>9} | train {tr:.3f} | test {te:.3f} | gap {tr - te:+.3f}")

Three readings of that table:

- **`max_depth=1`**: both scores low. The model is too simple to capture the pattern -
  **underfitting**, also called **high bias**.
- **`max_depth=None`**: train score is a perfect 1.000, test score is much lower.
  The model memorised noise - **overfitting**, also called **high variance**.
- Somewhere in the middle is the sweet spot. Finding it by eye in a printed table is
  painful, which is what the next plot is for.

## 2. Validation curve - how complex should the model be?

In [ ]:
from sklearn.model_selection import validation_curve

depths = np.arange(1, 21)
train_scores, val_scores = validation_curve(
    DecisionTreeClassifier(random_state=42), X, y,
    param_name="max_depth", param_range=depths,
    cv=5, scoring="accuracy")

train_mean, val_mean = train_scores.mean(axis=1), val_scores.mean(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(depths, train_mean, "o-", label="training score", color="#4C78A8")
plt.plot(depths, val_mean, "o-", label="cross-validation score", color="#F58518")
plt.fill_between(depths, val_scores.min(axis=1), val_scores.max(axis=1),
                 alpha=.15, color="#F58518")
plt.axvline(depths[val_mean.argmax()], ls="--", c="grey",
            label=f"best depth = {depths[val_mean.argmax()]}")
plt.xlabel("max_depth  (model complexity ->)")
plt.ylabel("accuracy")
plt.title("Validation curve: the gap opens as the tree gets deeper")
plt.legend(); plt.grid(alpha=.3)
plt.show()

**How to read it.** The blue line climbs towards 1.0 - a deep enough tree always memorises
the training data. The orange line peaks and then flattens or falls. The distance between
the two lines *is* the variance. Pick the depth at the orange peak, not the blue one.

## 3. Learning curve - would more data help?

This is the question that decides whether you go collect more rows or go change the model.

In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.ensemble import RandomForestClassifier

def plot_learning_curve(estimator, X, y, title, ax):
    sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=5, scoring="accuracy",
        train_sizes=np.linspace(0.1, 1.0, 8))
    ax.plot(sizes, train_scores.mean(axis=1), "o-", label="training", color="#4C78A8")
    ax.plot(sizes, val_scores.mean(axis=1), "o-", label="validation", color="#F58518")
    ax.set_title(title)
    ax.set_xlabel("training examples"); ax.set_ylabel("accuracy")
    ax.grid(alpha=.3); ax.legend(); ax.set_ylim(0.5, 1.05)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_learning_curve(DecisionTreeClassifier(max_depth=1, random_state=42), X, y,
                    "Underfit (max_depth=1)", axes[0])
plot_learning_curve(RandomForestClassifier(random_state=42), X, y,
                    "Overfit (unpruned forest)", axes[1])
plt.tight_layout()
plt.show()

**The decision rule:**

| What you see | Diagnosis | What to do |
|---|---|---|
| Both curves flat and **low**, close together | underfitting / high bias | more features, a more flexible model, less regularisation |
| Big **gap**, validation still rising | overfitting / high variance | **more data will help**, or regularise |
| Big gap, validation curve **flat** | overfitting, data-saturated | more data will *not* help - simplify or regularise instead |

That last row is the expensive mistake: paying to collect ten times more rows when the
curve had already flattened.

## 4. Seeing bias and variance directly

A regression example makes the geometry obvious.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

rng = np.random.RandomState(0)
xs = np.sort(rng.rand(30)) * 6
ys = np.sin(xs) + rng.randn(30) * 0.25
grid = np.linspace(0, 6, 300)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, deg, title in zip(axes, [1, 4, 20],
                          ["Underfit (degree 1)", "Good fit (degree 4)", "Overfit (degree 20)"]):
    m = make_pipeline(PolynomialFeatures(deg), LinearRegression()).fit(xs.reshape(-1, 1), ys)
    ax.scatter(xs, ys, s=25, color="#333333", zorder=3, label="data")
    ax.plot(grid, m.predict(grid.reshape(-1, 1)), color="#E45756", lw=2, label="model")
    ax.plot(grid, np.sin(grid), "--", color="grey", lw=1, label="truth")
    ax.set_ylim(-2, 2); ax.set_title(title); ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Left: the model is a straight line, it cannot bend - **bias**.
Right: the model chases every noisy point and swings wildly between them - **variance**.
Middle: it tracks the grey truth line. That is what you are aiming for every time.

---
## Exercises

1. Redo the validation curve for `RandomForestClassifier` over `n_estimators`
   (`[10, 25, 50, 100, 200, 400]`). Does the gap widen the way it did for a single tree?
   Explain the difference.
2. Run a validation curve over `min_samples_leaf` for a decision tree
   (`[1, 2, 5, 10, 20, 50]`). Which direction on that axis means *more* complexity?
3. Plot a learning curve for a `max_depth=3` tree. Which of the three rows in the table
   above does it match, and what would you do next?
4. In the polynomial plot, change the noise from `0.25` to `0.6`. At what degree does
   overfitting become visible now? What does that say about noisy data?
5. **Stretch:** write `diagnose(estimator, X, y)` that fits a learning curve and prints
   "underfitting", "overfitting" or "well-fit" based on the final train/validation gap
   and level. Test it on all three models above.